In [14]:
from rocqiomics import Rocqiomics
from rocqiomics.personal_utils import prepare_data_dicts

from monai.transforms import (
    Compose,
    ScaleIntensityd,
    NormalizeIntensityd,
    Spacingd,
    Resized
)

BASEPATH = r"C:\Users\ri54995\AppData\Local\Programs\Python\envs\radiomics\Lib\site-packages\rocqiomics\test_data"

In [15]:
# Custom function that generates data dicts from my usual folder structure.
# You don't need to use this, of course, as long as you generate the data_dicts with the required structure.
data_dicts, missing_img, missing_seg = prepare_data_dicts(
    base_dirpath=BASEPATH,
    mask_names=['Snickers'],
    modalities=['T2'],
    timepoints=['d1'],
)

data_dicts[0]

{'case_id': 'RGS021425_ph1',
 'image': 'C:\\Users\\ri54995\\AppData\\Local\\Programs\\Python\\envs\\radiomics\\Lib\\site-packages\\rocqiomics\\test_data\\Images\\d1_T2_RGS021425_ph1.nrrd',
 'mask': 'C:\\Users\\ri54995\\AppData\\Local\\Programs\\Python\\envs\\radiomics\\Lib\\site-packages\\rocqiomics\\test_data\\Masks\\Snickers\\d1_T2_RGS021425_ph1.seg.nrrd',
 'metadata': {'mask_name': 'Snickers', 'timepoint': 'd1', 'modality': 'T2'}}

In [20]:
voxel_based_settings = {
    'kernelRadius' : 1,
    'maskedKernel' : False,
    'initValue' : 0,
    'voxelBatch' : 2500
}

pipeline = Rocqiomics(
    data_dicts=data_dicts,
    preprocessing=Compose([
        NormalizeIntensityd(keys=['image']), # Standardize image intensity (subtract mean and divide by standard dev)
        ScaleIntensityd(keys=['image'], factor=99.0, minv=None, maxv=None), # Scale images by factor of 100 (99 + 1)
        Spacingd(keys=['image', 'mask'], pixdim=(0.125, 0.125, 1.0), mode=[3, 'nearest']), # Resample image and mask to specified voxel size
        Resized(keys=['image', 'mask'], spatial_size=(128, 128, -1), mode=['area', 'nearest'])
    ]),
    bin_width=1.0,
    voxel_based=True,
    voxel_based_settings=voxel_based_settings,
    force_2D=True,
    feature_classes=['glcm'],
    case_ids=['RGS021425_ph2'],
    save_results=False,
    save_dirpath='tutorial_results',
    save_by_columns=['mask_name', 'modality', 'timepoint'], # Save results to different files based on mask name, modality, and timepoint
)
pipeline.run_pipeline()
results = pipeline.get_results()

Rocqiomics INFO:	 Validating images and masks...


Rocqiomics INFO:	 Pipeline Initialized | Cases: 1 | Excluded cases: 0
Rocqiomics ERROR:	 Runtime error - RGS021425_ph2 - MemoryError((2500, 392, 392, 4), dtype('float64'))
